In [1]:
import ee
import geemap

In [2]:
PROJECT_ID = "solar-grail-510708-p8"

ee.Initialize(project=PROJECT_ID)

print("Earth Engine initialized successfully.")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine initialized successfully.


In [3]:
message = ee.String("Connected to Google Earth Engine")

print(message.getInfo())

Connected to Google Earth Engine


In [5]:
import geopandas as gpd

delhi_wards_gdf = gpd.read_file("C:/Users/Ā/Documents/python_class/dataset/wards_delhi_c.geojson")

delhi_wards_gdf.head()

,Ward_Name,Ward_No,geometry
0,DELHI CANTT CHARGE 1,CANT_1,"POLYGON ((77.13228 28.63154, 77.13644 28.62062..."
1,DELHI CANTT CHARGE 2,CANT_2,"POLYGON ((77.15429 28.62335, 77.15501 28.62228..."
2,DELHI CANTT CHARGE 4,CANT_4,"POLYGON ((77.15755 28.57578, 77.15672 28.57564..."
3,DELHI CANTT CHARGE 5,CANT_5,"POLYGON ((77.1348 28.57051, 77.13429 28.57048,..."
4,DELHI CANTT CHARGE 6,CANT_6,"POLYGON ((77.12157 28.59308, 77.12878 28.59029..."


In [6]:
print(
    "Number of Delhi wards:",
    len(delhi_wards_gdf)
)

Number of Delhi wards: 290


In [7]:
delhi_gdf = delhi_wards_gdf.dissolve()

In [8]:
delhi = geemap.geopandas_to_ee(
    delhi_gdf
)
delhi

In [9]:
Map = geemap.Map()

Map.centerObject(
    delhi,
    10
)

Map.addLayer(
    delhi,
    {"color": "white"},
    "Delhi Boundary"
)

Map

Map(center=[28.643030770927833, 77.11587615215892], controls=(WidgetControl(options=['position', 'transparent_…

In [10]:
landsat8 = ee.ImageCollection(
    "LANDSAT/LC08/C02/T1_L2"
)

landsat9 = ee.ImageCollection(
    "LANDSAT/LC09/C02/T1_L2"
)

landsat = landsat8.merge(
    landsat9
)

In [12]:
landsat_hot_season = (
    landsat
    .filterBounds(delhi)
    .filterDate(
        "2026-03-01",
        "2026-06-01"
    )
    .filter(
        ee.Filter.lt(
            "CLOUD_COVER",
            50
        )
    )
    .filter(
        ee.Filter.eq(
            "PROCESSING_LEVEL",
            "L2SP"
        )
    )
)

print(
    "Landsat scenes:",
    landsat_hot_season.size().getInfo()
)

Landsat scenes: 25


In [13]:
def prepare_landsat_lst(image):

    qa = image.select("QA_PIXEL")

    clear = (
        qa.bitwiseAnd(1 << 1).eq(0)
        .And(
            qa.bitwiseAnd(1 << 2).eq(0)
        )
        .And(
            qa.bitwiseAnd(1 << 3).eq(0)
        )
        .And(
            qa.bitwiseAnd(1 << 4).eq(0)
        )
        .And(
            qa.bitwiseAnd(1 << 5).eq(0)
        )
    )

    lst_c = (
        image
        .select("ST_B10")
        .multiply(0.00341802)
        .add(149.0)
        .subtract(273.15)
        .rename("LST_C")
    )

    return (
        lst_c
        .updateMask(clear)
        .copyProperties(
            image,
            ["system:time_start"]
        )
    )

In [14]:
landsat_lst = landsat_hot_season.map(
    prepare_landsat_lst
)

In [15]:
lst = (
    landsat_lst
    .median()
    .clip(delhi)
)

In [16]:
lst_vis = {
    "min": 20,
    "max": 45,
    "palette": [
        "navy",
        "blue",
        "cyan",
        "yellow",
        "orange",
        "red",
        "darkred"
    ]
}

In [17]:
Map = geemap.Map()

Map.centerObject(
    delhi,
    10
)

Map.addLayer(
    lst,
    lst_vis,
    "Median LST: Mar–May 2026"
)

Map.addLayer(
    delhi,
    {"color": "white"},
    "Delhi Boundary"
)

Map

Map(center=[28.643030770927698, 77.11587615215902], controls=(WidgetControl(options=['position', 'transparent_…